In [ ]:
import sys

sys.path.append("../../..")
from setup_figs import clean_names, pd, sns, plt, stats, pd

In [ ]:
fi = pd.read_parquet("0.parquet")
fi = clean_names(fi[fi.split == "test"])
spearman = pd.read_parquet("1.parquet")
spearman = clean_names(spearman[spearman.split == "test"])
weights = clean_names(pd.read_parquet("2.parquet"))
weights["AbsWeight"] = weights.Weight.abs()

In [ ]:
id_cols = [
    "trainer.model_builder.param",
    "trainer.representations.noise_level",
    "trainer.representations.seed",
]
hue_order = ["MLEM", "FR-RSA-I"]

In [ ]:
features_fi = (
    fi[fi["trainer.representations.noise_level"] == 0]
    .groupby(["Feature", "trainer.model_builder.param"])["mean"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .groupby("trainer.model_builder.param")
    .head(4)[["Feature", "trainer.model_builder.param"]]
)

In [ ]:
features_weights = (
    weights[weights["trainer.representations.noise_level"] == 0]
    .groupby(["Feature", "trainer.model_builder.param"])
    .AbsWeight.mean()
    .sort_values(ascending=False)
    .reset_index()
    .groupby("trainer.model_builder.param")
    .head(4)[["Feature", "trainer.model_builder.param"]]
)

In [ ]:
features = pd.concat([features_fi, features_weights])[["Feature"]].drop_duplicates()

# FI

In [ ]:
g = sns.relplot(
    fi.merge(features),
    kind="line",
    x="trainer.representations.noise_level",
    y="mean",
    hue="Feature",
    hue_order=features.Feature,
    col="trainer.model_builder.param",
    col_order=hue_order,
    height=4.5,
    aspect=1.5,
    errorbar="sd",
    marker="o",
    markersize=6,
)
g.set_ylabels("Feature Importance")
g.set_xlabels("Artificial Noise Level")
g.set_titles("{col_name}")
g.refline(y=0, linestyle="--", color="black", linewidth=1)
sns.despine(trim=True)
plt.savefig("../../../paper/figs/BERT_RC_mean/artificial_noise/feature_importance.pdf")
plt.show()

# Weights

In [ ]:
g = sns.relplot(
    weights.merge(features),
    kind="line",
    x="trainer.representations.noise_level",
    y="Weight",
    hue="Feature",
    hue_order=features.Feature,
    col="trainer.model_builder.param",
    col_order=hue_order,
    height=4.5,
    aspect=1.5,
    errorbar="sd",
    marker="o",
    markersize=6,
)
g.set_xlabels("Artificial Noise Level")
g.set_titles("{col_name}")
# g.refline(y=0, linestyle="--", color="black", linewidth=1)
sns.despine(trim=True)
plt.savefig("../../../paper/figs/BERT_RC_mean/artificial_noise/weights.pdf")
plt.show()

# Training duration

In [ ]:
ax = sns.lineplot(
    weights,
    x="trainer.representations.noise_level",
    y="training_duration",
    hue="trainer.model_builder.param",
    hue_order=hue_order,
    style="trainer.model_builder.param",
    markers=True,
    dashes=False,
    errorbar="sd",
)
ax.set_ylabel("Training Duration (s)")
ax.set_xlabel("Artificial Noise Level")
sns.move_legend(ax, "upper right", bbox_to_anchor=(1, 1.15), title=None)
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
sns.despine(trim=True)
plt.savefig("../../../paper/figs/BERT_RC_mean/artificial_noise/training_duration.pdf")
plt.show()

# Spearman

In [ ]:
ax = sns.lineplot(
    spearman,
    x="trainer.representations.noise_level",
    y="mean",
    hue="trainer.model_builder.param",
    hue_order=hue_order,
    style="trainer.model_builder.param",
    markers=True,
    dashes=False,
    errorbar="sd",
)
ax.set_ylim(-0.05, 1)
ax.set_ylabel(r"Test Spearman $\rho$")
ax.set_xlabel("Artificial Noise level")
sns.move_legend(ax, loc="best", title=None)
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
sns.despine(trim=True)
plt.savefig("../../../paper/figs/BERT_RC_mean/artificial_noise/spearman.pdf")
plt.show()

# Weighted $\tau$ and $L_2$

## Between MLEM and FR-RSA

In [ ]:
# weights_dist = weights.pivot(
#     index=[
#         "Feature",
#         "trainer.representations.seed",
#         "trainer.representations.noise_level",
#     ],
#     columns=["trainer.model_builder.param"],
#     values="Weight",
# ).reset_index()
# weights_dist = (
#     weights_dist.groupby(
#         [
#             "trainer.representations.seed",
#             "trainer.representations.noise_level",
#         ],
#     )
#     .apply(
#         lambda x: pd.Series(
#             {
#                 "Weighted $\\tau$": stats.weightedtau(
#                     x[hue_order[0]].abs(), x[hue_order[1]].abs()
#                 ).statistic,
#                 "L2": ((x[hue_order[0]] - x[hue_order[1]]) ** 2).sum() ** 0.5,
#             }
#         ),
#         include_groups=False,
#     )
#     .reset_index()
# )
# weights_dist["Type"] = "Weights"

# compare_fis = fi.pivot(
#     index=[
#         "Feature",
#         "trainer.representations.seed",
#         "trainer.representations.noise_level",
#     ],
#     columns=["trainer.model_builder.param"],
#     values="mean",
# ).reset_index()
# compare_fis = (
#     compare_fis.groupby(
#         ["trainer.representations.seed", "trainer.representations.noise_level"],
#     )
#     .apply(
#         lambda x: pd.Series(
#             {
#                 "Weighted $\\tau$": stats.weightedtau(
#                     x[hue_order[0]], x[hue_order[1]]
#                 ).statistic,
#                 "L2": ((x[hue_order[0]] - x[hue_order[1]]) ** 2).sum() ** 0.5,
#             }
#         ),
#         include_groups=False,
#     )
#     .reset_index()
# )
# compare_fis["Type"] = "FIs"

# compare = pd.concat([weights_dist, compare_fis])

In [ ]:
weights_dist = weights.pivot(
    index=[
        "Feature",
        "trainer.representations.seed",
        "trainer.representations.noise_level",
    ],
    columns=["trainer.model_builder.param"],
    values="Weight",
).reset_index()
weights_dist = (
    weights_dist.groupby(
        [
            "trainer.representations.seed",
            "trainer.representations.noise_level",
        ],
    )
    .apply(
        lambda x: ((x[hue_order[0]] - x[hue_order[1]]) ** 2).sum() ** 0.5,
        include_groups=False,
    )
    .reset_index(name="Frobenius distance")
)

In [ ]:
ax = sns.lineplot(
    weights_dist,
    x="trainer.representations.noise_level",
    y="Frobenius distance",
    marker="o",
    errorbar="sd",
)
ax.set_ylim(-0.1, 1)
ax.set_xlabel("Artificial Noise Level")
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
sns.despine(trim=True)
# plt.title("Weights distance between\nMLEM and FR-RSA-I")
plt.savefig("../../../paper/figs/BERT_RC_mean/artificial_noise/fro_mlem_frrsa.pdf")
plt.show()

In [ ]:
fis_dist = fi.pivot(
    index=[
        "Feature",
        "trainer.representations.seed",
        "trainer.representations.noise_level",
    ],
    columns=["trainer.model_builder.param"],
    values="mean",
).reset_index()
fis_dist = (
    fis_dist.groupby(
        ["trainer.representations.seed", "trainer.representations.noise_level"],
    )
    .apply(
        lambda x: stats.weightedtau(x[hue_order[0]], x[hue_order[1]]).statistic,
        include_groups=False,
    )
    .reset_index(name="Weighted $\\tau$")
)

In [ ]:
ax = sns.lineplot(
    fis_dist,
    x="trainer.representations.noise_level",
    y="Weighted $\\tau$",
    marker="o",
    errorbar="sd",
)
ax.set_ylim(-0.05, 1.1)
ax.set_xlabel("Artificial Noise Level")
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
# plt.title("FIs correlation between\nMLEM and FR-RSA-I", y=0.8)
sns.despine(trim=True)
plt.savefig(
    "../../../paper/figs/BERT_RC_mean/artificial_noise/weighted_tau_mlem_frrsa.pdf"
)
plt.show()

## between weights and FIs

In [ ]:
# weightedtau = weights.merge(fi, on=id_cols + ["Feature"])
# weightedtau = (
#     weightedtau.groupby(
#         id_cols,
#     )
#     .apply(
#         lambda x: stats.weightedtau(x.Weight.abs(), x["mean"].abs()).statistic,
#         include_groups=False,
#     )
#     .reset_index(name="Weighted $\\tau$")
# )

In [ ]:
# ax = sns.lineplot(
#     weightedtau,
#     x="trainer.representations.noise_level",
#     y="Weighted $\\tau$",
#     hue="trainer.model_builder.param",
#     hue_order=hue_order,
#     style="trainer.model_builder.param",
#     markers=True,
#     dashes=False,
#     errorbar="sd",
# )
# sns.move_legend(ax, loc="upper right", bbox_to_anchor=(1.1, 1.25), title=None)
# ax.set_xlabel("Artificial Noise Level")
# ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
# sns.despine(trim=True)
# plt.title(
#     "Weighted $\\tau$ between\nabsolute weights and FIs", x=-0.2, loc="left", pad=25
# )
# plt.savefig(
#     "../../../paper/figs/BERT_RC_mean/artificial_noise/weighted_tau_weights_fis.pdf"
# )
# plt.show()

## w.r.t. FIs/weights without noise

In [ ]:
weights_no_noise = weights.loc[
    weights["trainer.representations.noise_level"] == 0,
    ["Feature", "Weight"] + id_cols,
]
weights_no_noise = weights_no_noise.drop(columns=["trainer.representations.noise_level"])
weights_no_noise = weights_no_noise.rename(columns={"Weight": "Weight_no_noise"})
weights_dist = weights.merge(weights_no_noise)
weights_dist = (
    weights_dist.groupby(
        id_cols,
    )
    .apply(
        lambda x: ((x.Weight_no_noise - x.Weight) ** 2).sum() ** 0.5,
        include_groups=False,
    )
    .reset_index(name="Frobenius distance")
)

In [ ]:
ax = sns.lineplot(
    weights_dist,
    x="trainer.representations.noise_level",
    y="Frobenius distance",
    hue="trainer.model_builder.param",
    hue_order=hue_order,
    style="trainer.model_builder.param",
    markers=True,
    dashes=False,
    errorbar="sd",
)
ax.set_ylim(-0.1, 1.1)
ax.set_xlabel("Artificial Noise level")
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
sns.move_legend(
    ax, loc="center", bbox_to_anchor=(0.55, 1.1), title=None, frameon=True, ncols=2
)
# plt.title("Weights distance to noiseless estimation", pad=50)
sns.despine(trim=True)
plt.savefig("../../../paper/figs/BERT_RC_mean/artificial_noise/fro_noiseless.pdf")
plt.show()

In [ ]:
fi_no_noise = fi.loc[
    fi["trainer.representations.noise_level"] == 0,
    ["Feature", "mean"] + id_cols,
]
fi_no_noise = fi_no_noise.drop(columns=["trainer.representations.noise_level"])
fi_no_noise = fi_no_noise.rename(columns={"mean": "mean_no_noise"})
fis_dist = fi.merge(fi_no_noise)
fis_dist = (
    fis_dist.groupby(
        id_cols,
    )
    .apply(
        lambda x: stats.weightedtau(x.mean_no_noise, x["mean"]).statistic,
        include_groups=False,
    )
    .reset_index(name="Weighted $\\tau$")
)

In [ ]:
ax = sns.lineplot(
    fis_dist,
    x="trainer.representations.noise_level",
    y="Weighted $\\tau$",
    hue="trainer.model_builder.param",
    hue_order=hue_order,
    style="trainer.model_builder.param",
    markers=True,
    dashes=False,
    errorbar="sd",
)
ax.set_ylim(0, 1.05)
ax.set_xlabel("Artificial Noise level")
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
sns.move_legend(ax, loc="best", title=None, frameon=True, ncols=2)
# plt.title("FIs correlation with noiseless estimation", pad=15)
sns.despine(trim=True)
plt.savefig(
    "../../../paper/figs/BERT_RC_mean/artificial_noise/weighted_tau_noiseless.pdf"
)
plt.show()